## Objective

- Load a `.parquet` file of annotated data
- Count the number of unique decisions (`decision_id`)
- Display the total number of chunks

In [ ]:
import pandas as pd

# path to the Parquet file
PARQUET_FILE = "artifacts/tfidf/tfidf_chunks_train.parquet"  # not shipped — regenerated by this step (see DATA.md)

# load the Parquet file
df_count = pd.read_parquet(PARQUET_FILE)

# display number of unique decisions
nb_decisions = df_count["decision_id"].nunique()
print(f"Unique decisions: {nb_decisions}")

# display total number of chunks
nb_chunks = len(df_count)
print(f"Total chunks: {nb_chunks}")

# Full Training Pipeline for the JuriBERT Bi-Encoder + Attention + MLP

This script runs the complete pipeline to train a custom legal bi-encoder based on **JuriBERT** with: transformer backbone, multi-head attention layer, and MLP head.

**Main pipeline steps:**

1. **Data preparation and export**
   - Convert an Excel of `[chunk, article]` pairs to an SBERT-compatible `.jsonl` file.
   - Each entry contains: masked chunk text, original text, article text, labels, IDs, etc.

2. **Stratified dataset splitting**
   - **Train / validation / test** split by decision, to prevent information leakage between sets.

3. **InputExample construction**
   - Create `InputExample` objects:
     - For training (positive pairs only),
     - For validation/test (all pairs, with binary label).

4. **Custom module definitions**
   - **AttentionLayer**: PyTorch multi-head attention layer applied to token embeddings.
   - **TransformerWithAttention**: backbone combining tokenizer, JuriBERT, attention, and dense projection.
   - **MLPHead**: dense head applied to the global (sentence) embedding, to better adapt the vector space for legal similarity.

5. **Full model instantiation**
   - Enrich tokenizer with special tokens `[ARTICLE]`, `[DECISION]`.
   - Assemble: `TransformerWithAttention` -> MEAN pooling -> `MLPHead` in a `SentenceTransformer` ready for training.

6. **DataLoader preparation**
   - Load train/val/test sets into DataLoaders for PyTorch batching.

7. **Loss and metrics definition**
   - `MultipleNegativesRankingLoss` (ranking on positive examples).
   - Continuous evaluation via `BinaryClassificationEvaluator` during fit.

8. **Training and saving**
   - Launch training with periodic validation, automatic saving of the best complete model to `artifacts/model/bi_encoder_juribert`.
   - All custom modules (tokenizer, transformer, attention, pooling, MLP) are saved **properly and once**, ensuring reproducibility and lossless reload.

> **Goal**: Obtain a legal bi-encoder model, reusable for chunk-article similarity search, associating decision context with the applied legal article.

---

In [ ]:
# -*- coding: utf-8 -*-
"""
Bi-encoder JuriBERT + Attention + MLP

  tokenizer: JuriBERT/LegalTokenizer
  model: JuriBERT/juribert_base
  output: artifacts/model/bi_encoder_juribert  # not shipped — regenerated by this step (see DATA.md)

Goal: prepare data, train the bi-encoder, then save all modules
(transformer, attention, enriched tokenizer, pooling, and MLP)
properly and once.
"""

# ========== imports ==========================================================
import os, json
import pandas as pd

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

from transformers import RobertaTokenizerFast, AutoTokenizer, AutoModel
from sentence_transformers import (
    SentenceTransformer, InputExample, models
)
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.evaluation import BinaryClassificationEvaluator

# ========== 1. dataset preparation ===========================================

df_train = pd.read_parquet("artifacts/tfidf/tfidf_POS_chunks_train.parquet")
df_val = pd.read_parquet("artifacts/tfidf/tfidf_chunks_VALIDATION.parquet")

train_examples = [
    InputExample(texts=["[DECISION] " + row["chunk_text_masked"],
                        "[ARTICLE] " + row["article_text"]])
    for _, row in df_train.iterrows()
]

val_examples = [
    InputExample(texts=["[DECISION] " + row["chunk_text_masked"],
                        "[ARTICLE] " + row["article_text"]],
                 label=float(row["label"]))
    for _, row in df_val.iterrows()
]

# ========== 3. custom modules: attention & TransformerWithAttention ==========
class AttentionLayer(nn.Module):
    # multi-head attention layer on a sequence of embeddings
    # allows each token to attend to all others, enriching contextual representations
    def __init__(self, input_dim: int, num_heads: int = 8):
        super().__init__()
        self.attention = nn.MultiheadAttention(input_dim, num_heads)
        self.input_dim, self.num_heads = input_dim, num_heads

    def forward(self, x):
        # x shape: (seq_len, batch_size, embedding_dim)
        # self-attention recomputes each token based on all other tokens
        attn_out, _ = self.attention(x, x, x)
        # output has the same shape; each token embedding is now contextualized
        return attn_out

    # --- I/O ---
    def save(self, path):
        # save attention weights and config
        torch.save(self.state_dict(), os.path.join(path, "attention.pt"))
        json.dump({"input_dim": self.input_dim, "num_heads": self.num_heads},
                  open(os.path.join(path, "attention_config.json"), "w"))
    @classmethod
    def load(cls, path):
        # reload config and weights to restore the attention layer identically
        cfg = json.load(open(os.path.join(path, "attention_config.json")))
        obj = cls(cfg["input_dim"], cfg["num_heads"])
        obj.load_state_dict(torch.load(os.path.join(path, "attention.pt"), map_location="cpu"))
        return obj

class TransformerWithAttention(nn.Module):
    # main backbone: JuriBERT + multi-head attention + linear projection
    # handles the pipeline: tokenization -> JuriBERT -> attention -> dense+relu
    def __init__(self, transformer, attention, tokenizer):
        super().__init__()
        self.transformer, self.attention, self.tokenizer = transformer, attention, tokenizer
        # linear projection to transform each token embedding after attention
        self.projection = nn.Sequential(
            nn.Linear(transformer.config.hidden_size, transformer.config.hidden_size),
            nn.ReLU(),
        )

    def forward(self, features):
        # extract input_ids and attention_mask from the features dict
        ids, mask = features["input_ids"], features["attention_mask"]
        # pass through JuriBERT -> last_hidden_state (batch, seq, dim)
        x = self.transformer(ids, attention_mask=mask).last_hidden_state
        # permute to (seq, batch, dim) as expected by MultiheadAttention
        x = self.attention(x.permute(1, 0, 2)).permute(1, 0, 2)
        # dense+relu projection further refines each token embedding
        features["token_embeddings"] = self.projection(x)
        # at this point, token embeddings are contextualized and transformed
        return features

    def tokenize(self, texts, text_pair=None):
        # tokenization utility compatible with SBERT
        if isinstance(texts, str): texts = [texts]
        if isinstance(text_pair, str): text_pair = [text_pair]
        return self.tokenizer(
            texts, text_pair, padding=True, truncation=True, max_length=512,
            return_tensors="pt"
        )

    # --- I/O ---
    def save(self, path):
        # save entire custom backbone (weights, config, tokenizer, transformer, attention, projection)
        torch.save(self.state_dict(), os.path.join(path, "transformer_with_attention.pt"))
        self.tokenizer.save_pretrained(os.path.join(path, "juribert_tokenizer"))
        self.transformer.save_pretrained(os.path.join(path, "juribert_model"))
        self.attention.save(path)
        torch.save(self.projection.state_dict(), os.path.join(path, "projection.pt"))

    @classmethod
    def load(cls, path):
        # reload each component identically for inference or retraining
        trf = AutoModel.from_pretrained(os.path.join(path, "juribert_model"))
        tok = AutoTokenizer.from_pretrained(os.path.join(path, "juribert_tokenizer"))
        attn = AttentionLayer.load(path)
        obj = cls(trf, attn, tok)
        obj.projection.load_state_dict(torch.load(os.path.join(path, "projection.pt"), map_location="cpu"))
        obj.load_state_dict(torch.load(os.path.join(path, "transformer_with_attention.pt"), map_location="cpu"))
        return obj

class MLPHead(nn.Module):
    # dense head (MLP) applied to the global embedding (after pooling)
    # refines the vector space for legal similarity
    def __init__(self, input_dim: int, hidden_dim: int = 512, output_dim: int | None = None):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim or input_dim)
        )

    def forward(self, features, **_):
        # apply MLP to the sentence embedding (pooling output)
        features["sentence_embedding"] = self.mlp(features["sentence_embedding"])
        return features

    # --- I/O ---
    def save(self, path):
        # save MLP weights and config
        torch.save(self.state_dict(), os.path.join(path, "mlp_head.pt"))
        cfg = {"input_dim": self.mlp[0].in_features,
               "hidden_dim": self.mlp[0].out_features,
               "output_dim": self.mlp[-1].out_features}
        json.dump(cfg, open(os.path.join(path, "mlp_head_config.json"), "w"))

    @classmethod
    def load(cls, path):
        # reload MLP (weights and config)
        cfg = json.load(open(os.path.join(path, "mlp_head_config.json")))
        obj = cls(cfg["input_dim"], cfg["hidden_dim"], cfg["output_dim"])
        obj.load_state_dict(torch.load(os.path.join(path, "mlp_head.pt"), map_location="cpu"))
        return obj

# ========== 4. full model instantiation ======================================
tokenizer_path = "JuriBERT/LegalTokenizer"  # external pretrained JuriBERT model, not shipped - see DATA.md
model_path = "JuriBERT/juribert_base"  # external pretrained JuriBERT model, not shipped - see DATA.md

tokenizer = RobertaTokenizerFast.from_pretrained(tokenizer_path)
tokenizer.add_tokens(["[ARTICLE]", "[DECISION]"])  # enrich vocabulary with special tokens

juribert = AutoModel.from_pretrained(model_path)
juribert.resize_token_embeddings(len(tokenizer))  # resize embedding matrix for enriched vocabulary

attention_layer = AttentionLayer(juribert.config.hidden_size)
word_embed_model = TransformerWithAttention(juribert, attention_layer, tokenizer)
pooling_model = models.Pooling(juribert.config.hidden_size, pooling_mode="mean")
mlp_head = MLPHead(juribert.config.hidden_size, hidden_dim=512)

# assemble all modules into the SBERT bi-encoder
# each module is called sequentially: encoding -> attention -> projection -> pooling -> MLP
bi_encoder = SentenceTransformer(modules=[word_embed_model, pooling_model, mlp_head])

# ========== 5. dataloaders ===================================================
class PairDataset(Dataset):
    # dataset adapter for PyTorch DataLoaders to handle InputExamples
    def __init__(self, examples): self.examples = examples
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx): return self.examples[idx]

# dataloaders for batched training
train_loader = DataLoader(PairDataset(train_examples), shuffle=True, batch_size=16)
val_loader = DataLoader(PairDataset(val_examples), shuffle=False, batch_size=16)

# ========== 6. loss & evaluators =============================================
# multiple negatives ranking loss works well for the bi-encoder
train_loss = MultipleNegativesRankingLoss(bi_encoder)
val_evaluator = BinaryClassificationEvaluator.from_input_examples(val_examples, name="val")

In [ ]:

# ========== 7. training & single save ========================================
OUTPUT_DIR = "artifacts/model/bi_encoder_juribert"  # not shipped — regenerated by this step (see DATA.md)

# fit() trains the bi-encoder, keeping the best model on validation in memory
bi_encoder.fit(
    train_objectives=[(train_loader, train_loss)],
    epochs=3,
    warmup_steps=100,
    evaluator=val_evaluator,
    evaluation_steps=400,
    output_path=OUTPUT_DIR,  # SentenceTransformer.save() is called internally
    save_best_model=True,
    show_progress_bar=True,
)
print(f"Training complete, model saved to: {OUTPUT_DIR}")

# Loading a Custom Bi-Encoder (JuriBERT + Attention + MLP)

This block **reconstructs the complete bi-encoder** from individually saved modules after training.

- Paths are defined for:
  the custom backbone (JuriBERT + attention + projection),
  the pooling layer (token embedding aggregation),
  the MLP head (final dense projection).

- Each module is reloaded via its dedicated `.load()` method:
  `TransformerWithAttention.load()` for the backbone,
  `models.Pooling.load()` for pooling,
  `MLPHead.load()` for the MLP head.

- Finally, all modules are **assembled into a SentenceTransformer**, ready for inference or re-evaluation.

> **Note**: This method recovers *exactly* the same architecture and weights as at the end of training, even for custom models.

---

In [ ]:
import os
from sentence_transformers import SentenceTransformer, models

# paths to saved modules
model_dir = "artifacts/model/bi_encoder_juribert"  # not shipped — regenerated by this step (see DATA.md)
backbone_path_bis = os.path.join(model_dir, "0_TransformerWithAttention")
pooling_path_bis = os.path.join(model_dir, "1_Pooling")
mlp_head_path_bis = os.path.join(model_dir, "2_MLPHead")

# load custom backbone, pooling, and MLP head
backbone_bis = TransformerWithAttention.load(backbone_path_bis)
pooling_bis = models.Pooling.load(pooling_path_bis)
mlp_head_bis = MLPHead.load(mlp_head_path_bis)

# assemble the bi-encoder
bi_encoder_bis = SentenceTransformer(modules=[backbone_bis, pooling_bis, mlp_head_bis])